In [1]:
import pandas as pd
data = pd.read_csv(r"D:\ansci-4040-fall-2026\jl4937_6040-project-1\dataset\dataset_for_model_buliding_2.csv", low_memory=False)


In [2]:
# Count null values by column
null_counts = data.isna().sum()

# Count zero values in numeric columns
zero_counts = data.select_dtypes(include="number").eq(0).sum()

# Display counts
summary = pd.DataFrame({
    "null_count": null_counts,
    "zero_count": zero_counts.reindex(data.columns, fill_value=0)
})

print(summary)
print(f"\nTotal null values: {data.isna().sum().sum()}")
print(f"Total zero values: {zero_counts.sum()}")

                        null_count  zero_count
AnimalId                         0           0
LactationNumber                  0           0
DaysInMilk                       0           0
ReproductionStatus               0           0
EventDate                        0           0
Avgmilkflow                      0           0
Flow30_60Session                 0           0
YieldFirst2Min_Session           0           0
YieldSession                     0           0
DurationSession_sec              0           0
milking                          0           0

Total null values: 0
Total zero values: 0


In [3]:
# Convert EventDate to datetime and sort from earliest to latest
data["EventDate"] = pd.to_datetime(data["EventDate"], errors="coerce")

data = (
    data.sort_values("EventDate", ascending=True)
        .reset_index(drop=True)
)

print(data[["EventDate"]].head())
print(data[["EventDate"]].tail())

   EventDate
0 2019-06-14
1 2019-06-14
2 2019-06-14
3 2019-06-14
4 2019-06-14
         EventDate
6645167 2021-10-24
6645168 2021-10-24
6645169 2021-10-24
6645170 2021-10-24
6645171 2021-10-24


In [4]:
# Split the time-ordered data chronologically without shuffling
n = len(data)

train_end = int(n * 0.6)
validation_end = train_end + int(n * 0.2)

data_train = data.iloc[:train_end].copy()
data_validation = data.iloc[train_end:validation_end].copy()
data_test = data.iloc[validation_end:].copy()

print(f"Training set:   {len(data_train):,} rows ({len(data_train) / n:.1%})")
print(f"Validation set: {len(data_validation):,} rows ({len(data_validation) / n:.1%})")
print(f"Test set:       {len(data_test):,} rows ({len(data_test) / n:.1%})")

print("\nDate ranges:")
print(f"Training:   {data_train['EventDate'].min()} to {data_train['EventDate'].max()}")
print(f"Validation: {data_validation['EventDate'].min()} to {data_validation['EventDate'].max()}")
print(f"Test:       {data_test['EventDate'].min()} to {data_test['EventDate'].max()}")

Training set:   3,987,103 rows (60.0%)
Validation set: 1,329,034 rows (20.0%)
Test set:       1,329,035 rows (20.0%)

Date ranges:
Training:   2019-06-14 00:00:00 to 2020-12-14 00:00:00
Validation: 2020-12-14 00:00:00 to 2021-05-21 00:00:00
Test:       2021-05-21 00:00:00 to 2021-10-24 00:00:00


In [ ]:
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score

# Check whether AnimalId is predictable as a repeated class
animal_id_count = data["AnimalId"].nunique()
print(f"Unique AnimalId values: {animal_id_count:,}")
print(f"Rows: {len(data):,}")
print(f"Uniqueness ratio: {animal_id_count / len(data):.2%}")

# Create model features without AnimalId
def make_features(frame):
    features = frame.drop(columns=["AnimalId"]).copy()

    features["EventDate"] = pd.to_datetime(features["EventDate"])
    features["EventYear"] = features["EventDate"].dt.year
    features["EventMonth"] = features["EventDate"].dt.month
    features["EventDay"] = features["EventDate"].dt.day
    features["EventDayOfWeek"] = features["EventDate"].dt.dayofweek
    features = features.drop(columns=["EventDate"])

    return pd.get_dummies(features, columns=["ReproductionStatus"], dtype=np.float32)

X_train = make_features(data_train)
X_validation = make_features(data_validation)

# Ensure identical columns in both sets
X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)

# Encode AnimalId using training labels
train_ids, class_values = pd.factorize(data_train["AnimalId"])
class_to_number = {value: i for i, value in enumerate(class_values)}

validation_ids = data_validation["AnimalId"].map(class_to_number)
known_validation = validation_ids.notna()

y_train = train_ids
y_validation = validation_ids[known_validation].astype(int)

X_validation = X_validation.loc[known_validation]

# Sample for practical training on this very large dataset
rng = np.random.default_rng(42)

max_train_rows = min(200_000, len(X_train))
max_validation_rows = min(50_000, len(X_validation))

train_indices = rng.choice(len(X_train), max_train_rows, replace=False)
validation_indices = rng.choice(len(X_validation), max_validation_rows, replace=False)

X_train_sample = X_train.iloc[train_indices]
y_train_sample = y_train[train_indices]

X_validation_sample = X_validation.iloc[validation_indices]
y_validation_sample = y_validation.iloc[validation_indices]

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=100,
        solver="lbfgs",
        multi_class="auto",
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=42
    )
}

results = []

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")

    try:
        model.fit(X_train_sample, y_train_sample)

        predictions = model.predict(X_validation_sample)
        probabilities = model.predict_proba(X_validation_sample)

        accuracy = accuracy_score(y_validation_sample, predictions)

        k = min(5, probabilities.shape[1])
        top_5_accuracy = top_k_accuracy_score(
            y_validation_sample,
            probabilities,
            k=k,
            labels=np.arange(len(class_values))
        )

        results.append({
            "model": model_name,
            "accuracy": accuracy,
            "top_5_accuracy": top_5_accuracy
        })

        print(f"Accuracy:     {accuracy:.4f}")
        print(f"Top-{k} accuracy: {top_5_accuracy:.4f}")

    except Exception as error:
        print(f"{model_name} failed: {error}")

results_df = pd.DataFrame(results).sort_values(
    "top_5_accuracy",
    ascending=False
)

results_df